# TAVI is under active development, functions may change without notice. If encounter any errors, contact Kyle Ma (maq1@ornl.gov)

In [1]:
from tavi.library.Instrument.instrument import Instrument, InstrumentCatalog
from tavi.library.experiment.experiment import Experiment
from tavi.library.experiment.utilities import mantid_to_spice
from tavi.library.tas.triple_axis import TAS
from tavi.library.plugin.hb1a_plugins import VERITAS
from tavi.library.fit.fit import FitPackage, ModelName
from pathlib import Path

### Load data, we can load sample and UB from a .dat file. Make sure the sample_path and data_path are correct. If you prefer to load sample manually, uncomment the section below.

In [2]:
# if want to mannually create sample or UB.

# from tavi.library.experiment.utilities import spice_to_mantid
# from tavi.library.geometry.oriented_lattice import OrientedLattice

# ol = OrientedLattice(a=1, b =1, c = 1, alpha=90, beta = 90, gamma = 90)
# spice_ub = np.array([[-0.144255,-0.072128,-0.057896],
#                      [0.184002,0.092001,-0.045390],
#                      [0.000000,-0.202484,-0.000000]])
# ol.UB = spice_to_mantid(spice_ub, version="old")
# sample = Sample(ol)

# experiment = Experiment()
# tas = TAS(Instrument(instrument_catalog=InstrumentCatalog.HB1A4C), sample, experiment, plugin=VERITAS)
# data_path = Path.cwd().parent.parent.joinpath("IPTS-33477", "exp1046", "Datafiles")
# tas.experiment.load_folder(data_path)
# print(len(tas.experiment.tavi_data.raw_scans))

In [3]:
experiment = Experiment()
sample_path = Path.cwd().parent.parent.joinpath("IPTS-38327", "exp1100", "UBConf", "UBmatrix_4K.dat")
sample = experiment.create_sample(sample_path)
sample.mosaic_h = 0
sample.mosaic_v = 0

In [4]:
tas = TAS(Instrument(instrument_catalog=InstrumentCatalog.HB1A4C), sample, experiment, plugin=VERITAS)
data_path = Path.cwd().parent.parent.joinpath("IPTS-38327", "exp1100", "Datafiles")
tas.experiment.load_folder(data_path)
print(len(tas.experiment.tavi_data.raw_scans))

No readable UBConf file at /gpfs/neutronsfs/instruments-hfir/HB1A/IPTS-38327/exp1100/UBConf/UB27Sep2026_111501AM.ini; loading /gpfs/neutronsfs/instruments-hfir/HB1A/IPTS-38327/exp1100/Datafiles/HB1A_exp1100_scan0001.dat without UB metadata.
No readable UBConf file at /gpfs/neutronsfs/instruments-hfir/HB1A/IPTS-38327/exp1100/UBConf/UB27Sep2026_111501AM.ini; loading /gpfs/neutronsfs/instruments-hfir/HB1A/IPTS-38327/exp1100/Datafiles/HB1A_exp1100_scan0002.dat without UB metadata.
No readable UBConf file at /gpfs/neutronsfs/instruments-hfir/HB1A/IPTS-38327/exp1100/UBConf/UB27Sep2026_111501AM.ini; loading /gpfs/neutronsfs/instruments-hfir/HB1A/IPTS-38327/exp1100/Datafiles/HB1A_exp1100_scan0003.dat without UB metadata.
No readable UBConf file at /gpfs/neutronsfs/instruments-hfir/HB1A/IPTS-38327/exp1100/UBConf/UB27Sep2026_111501AM.ini; loading /gpfs/neutronsfs/instruments-hfir/HB1A/IPTS-38327/exp1100/Datafiles/HB1A_exp1100_scan0004.dat without UB metadata.
No readable UBConf file at /gpfs/neu

447


### Set a number of scans to take a look at low temperature data.

In [5]:
LowT_list = list(range(61, 77+1)) + list(range(111, 240+1))
tas.browse(LowT_list, show_fits = False, save_figure = "browse_scans_4K.pdf")

(None, None, None)

### Browsing the data, if there are some scans doesn't look correct, we can remove them. It's also good to confirm the UB matrices are correct from what's loaded and what's calcualted

In [6]:
remove_list = [61, 65, 114, 124, 126, 128, 130, 137, 139, 141, 143, 148, 171, 177, 184, 191, 193, 195, 197, 198, 203, 205, 206, 230, 237, 239]
for num in remove_list:
    LowT_list.remove(num)
peaks = [
    dp
    for i in LowT_list
    for dp in tas.experiment.get_closest_to_center_data_point({"scan_num": i}, FitPackage.lmfit, [(ModelName.Gaussian, dict(guess=True))])
]
print(f"Compare \n {mantid_to_spice(tas.ub(peaks = peaks, reset_ub=False), version="new")} \n with data.")
# print(tas.ub(peaks = peaks))
print(mantid_to_spice(sample.ol.UB, version = "new"))

Compare 
 [[ 0.18210516  0.09100876 -0.03175077]
 [ 0.08322118  0.03671151  0.06941364]
 [-0.00236338 -0.17452123 -0.00194983]] 
 with data.
[[ 0.1820519   0.09098234 -0.03174121]
 [ 0.08319709  0.03670086  0.06939339]
 [-0.00236309 -0.17447031 -0.00194945]]


### Optional to visualizeresolution ellipses

In [7]:
tas.browse_resolution_ellipse(LowT_list, xlabel = "Q_para", ylabel = "Qperp", resolution_frame = "local", ellipse_axes = (0,1), save_figure = "LowT_ellipses.pdf")

### Specify a Gaussian and linear background to fit the scans, normalize them to time or mcu, display coherent resolution bar and save the figure

In [8]:
hkls, fit_results, res_4ds = tas.browse(LowT_list, 
                                        show_fits=True, 
                                        model_dict=[(ModelName.Gaussian, dict(prefix="g",guess = True)),
                                                    (ModelName.Linear, dict(prefix="l",guess=True))], 
                                        resolution_frame = "local",
                                        show_resolution_bar=True,
                                        show_components=True,
                                        projection_axis=1,
                                        normalize = "mcu",
                                        multiply_factor = 60.0,
                                        coh_bar = True,
                                       save_figure = "Peak_fitting_with_resolution_bar_4K.pdf")

### Set a number of scans to take a look at high temperature data.

In [9]:
Data_list = list(range(80, 94+1)) + list(range(250, 356+1))
tas.browse(Data_list, show_fits = False, save_figure = "test_browse_scans_100K.pdf")

(None, None, None)

### Overplot low T and high T data without normalize to same counting time (if they are different).

In [10]:
high_T = list(range(80, 94+1)) + list(range(250, 355+1))
print(len(LowT_list), len(high_T))
tas.browse(LowT_list, show_fits = False, normalize = "mcu", multiply_factor = 60.0, save_figure = "Overplot_LTvsHT.pdf", background = high_T)

121 121


(None, None, None)

### Specify a Gaussian and linear background to fit the scans, normalize them to time (or mcu), display coherent resolution bar and save figure.

In [11]:
hkls, fit_results, res_4ds = tas.browse(LowT_list, 
                                        show_fits=True, 
                                        model_dict=[(ModelName.Gaussian, dict(prefix="g",guess = True)),
                                                    (ModelName.Linear, dict(prefix="l",guess=True))], 
                                        resolution_frame = "local",
                                        show_resolution_bar=True,
                                        show_components=True,
                                        projection_axis=1,
                                        normalize = "mcu",
                                        multiply_factor = 60.0,
                                        coh_bar = True,
                                       save_figure = "Peak_fitting_with_resolution_bar_LTvsHT.pdf",
                                       background = high_T) # specify high_T as background

### Use high_T list as background to generate hkls, fit_results and res_4ds

In [12]:
hkls_bkg, fit_results_bkg, res_4ds_bkg = tas.browse(high_T, 
                                        show_fits=True, 
                                        model_dict=[(ModelName.Gaussian, dict(prefix="g",guess = True, set = {"amplitude": dict(min = 0)})),
                                                    (ModelName.Linear, dict(prefix="l",guess=True))], 
                                        resolution_frame = "local",
                                        show_resolution_bar=True,
                                        show_components=True,
                                        projection_axis=1,
                                        normalize = "mcu",
                                        multiply_factor = 60.0,
                                        coh_bar = True,
                                       save_figure = "Peak_fitting_with_resolution_bar_100K.pdf",
                                       )
# If some measurements has no peak, the fitting will fail. We need to remove these scans.
fit_results_bkg = tas.plugin.remove_peak(high_T, fit_results_bkg, no_peak = [88,250,252, 259, 261, 264, 318, 337, 341, 342, 344, 348, 351])

### Give the exported file a title, replace "sample results" if needed, save the file to "test.int" 

In [13]:
export=tas.plugin.export_intensity("Single crystal data of SrV5O11 (HB1A)", hkls, fit_results, res_4ds, ax = "s1", save_to_file = "Peaks_4K.int", wavelength = 2.37815)

In [14]:
export=tas.plugin.export_intensity("Single crystal data of SrV5O11 (HB1A)", hkls_bkg, fit_results_bkg, res_4ds_bkg, ax = "s1", save_to_file = "Peaks_100K.int", wavelength = 2.37815)

In [15]:
export=tas.plugin.export_intensity("Single crystal data of SrV5O11 (HB1A)", hkls, fit_results, res_4ds, ax = "s1", save_to_file = "Magnetic.int", wavelength = 2.37815, 
                                   background_results=fit_results_bkg,wavevector = [0,0,0])